<h3><b>1. 라이브러리 가져오기 

필요 라이브러리 설치하기 

In [13]:
!pip install python-barcode

라이브러리 가져오기

In [5]:
import barcode
import cv2

바코드 패키지에서 사용가능한 것 확인하기

In [6]:
dir(barcode)

['BarcodeNotFoundError',
 'BinaryIO',
 'CODABAR',
 'Code128',
 'Code39',
 'Dict',
 'EAN13',
 'EAN13_GUARD',
 'EAN14',
 'EAN8',
 'EAN8_GUARD',
 'Gs1_128',
 'ISBN10',
 'ISBN13',
 'ISSN',
 'ITF',
 'JAN',
 'Optional',
 'PROVIDED_BARCODES',
 'PZN',
 'UPCA',
 'Union',
 '__BARCODE_MAP',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'base',
 'charsets',
 'codabar',
 'codex',
 'ean',
 'errors',
 'generate',
 'get',
 'get_barcode',
 'get_barcode_class',
 'get_class',
 'isxn',
 'itf',
 'os',
 'upc',
 'version',
 'writer']

<h3><b>2. ISBN-13 바코드 생성하기 

ISBN-13은 도서 관련 국제 순서 표기 번호이다. 

ISBN-13 관련 라이브러리 가져오기 및 ImageWriter 가져오기 

In [1]:
from barcode import ISBN13
from barcode.writer import ImageWriter
import cv2

c:\Users\jjang\anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
c:\Users\jjang\anaconda3\lib\site-packages\numpy\.libs\libopenblas.FB5AE2TYXYH2IJRDKGDGQ3XBKLKTF43H.gfortran-win_amd64.dll
c:\Users\jjang\anaconda3\lib\site-packages\numpy\.libs\libopenblas64__v0.3.23-246-g3d31191b-gcc_10_3_0.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


number를 문자열로 정의하기

In [2]:
num = '978456752421'
num

'978456752421'

ISBN-13 클래스 초기화 하기 

In [3]:
book_barcode = ISBN13(num, writer=ImageWriter())
book_barcode

<InternationalStandardBookNumber13('9784567524216')>

파일 생성 및 저장 위치 설정하기 

In [4]:
outputPath = 'C:/Users/jjang/Desktop/mechArm_jupyter/'

In [5]:
output = outputPath + 'isbn13_barcode1'
output

'C:/Users/jjang/Desktop/mechArm_jupyter/isbn13_barcode1'

바코드 이미지 저장 및 확인하기 

In [6]:
book_barcode.save(output)

a = cv2.imread('isbn13_barcode1.png')
cv2.imshow('barcode', a)

<h3><b>3. 카메라로 바코드 인식 바운딩 박스 생성하기 

카메라 불러오기

In [16]:
import cv2
from pyzbar.pyzbar import decode

바코드 인식 함수 생성하기 

In [17]:
def barcodeReader(frame):
    # 입력된 프레임에서 바코드 찾기 
    detected_barcodes = decode(frame)

    # 바코드 식별을 위한 최소 크기 설정 
    min_width = 10  # 바코드 영역의 최소 너비 임계값 
    min_height = 10  # 바코드 영역의 최소 높이 임계값 
    
    if not detected_barcodes:
        print("Barcode Not Detected or your barcode is blank/corrupted!")
    else:
        # 바코드 인식 시, 바코드의 위치와 크기 가져오기 
        for barcode in detected_barcodes:
            (x, y, w, h) = barcode.rect
            
            # 바코드 너비와 높이가 최소 임계값보다 큰지 확인 >> 유효한 바코드인지 확인하기 
            if w > min_width and h > min_height:
                # 바코드에 바운딩 박스 생성 
                cv2.rectangle(frame, (x-10, y-10), (x + w+10, y + h+10), (255, 0, 0), 2)
                if barcode.data:
                    barcode_data = barcode.data.decode("utf-8")
                    barcode_type = barcode.type
                    print("Barcode Data:", barcode_data)
                    print("Barcode Type:", barcode_type)

    cv2.imshow("Barcode Detection", frame)

결과 확인하기 

In [19]:
if __name__ == "__main__":
    # 카메라 작동시키기 
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()

        if ret:
            barcodeReader(frame)

        # 'q'를 누르면 종료
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your ba

<h3><b>4. 바코드 인식 결과 값에 따른 MC-270 움직임 제어하기 

In [21]:
import cv2
from pyzbar.pyzbar import decode
from pymycobot.mecharm import MechArm
import time

In [22]:
mc = MechArm("com3", 115200)

바코드 인식 후 MC-270 움직임 제어 함수 

In [28]:
def move_mechArm(barcode_data):
    if barcode_data == "9784567524216":
        print("Moving!")
        mc.send_angles([28.3, 55.98, -8.43, -1.23, 20.83, 1.58], 30)
        time.sleep(2)
        mc.send_angles([-33.66, -28.38, -8.43, 0.35, 37.88, 1.58], 30)
        time.sleep(2)
        mc.send_angles([0, 0, 0, 0, 0, 0], 30)
        
    else:
        print("Not Found the barcode")

바코드 인식 함수 

In [31]:
def barcodeReader(frame):
    # 입력된 프레임에서 바코드 찾기 
    detected_barcodes = decode(frame)

    # 바코드 식별을 위한 최소 크기 설정 
    min_width = 10  # 바코드 영역의 최소 너비 임계값 
    min_height = 10  # 바코드 영역의 최소 높이 임계값 
    
    if not detected_barcodes:
        print("Barcode Not Detected or your barcode is blank/corrupted!")
    else:
        # 바코드 인식 시, 바코드의 위치와 크기 가져오기 
        for barcode in detected_barcodes:
            (x, y, w, h) = barcode.rect
            
            # 바코드 너비와 높이가 최소 임계값보다 큰지 확인 >> 유효한 바코드인지 확인하기 
            if w > min_width and h > min_height:
                # 바코드에 바운딩 박스 생성 
                cv2.rectangle(frame, (x-10, y-10), (x + w+10, y + h+10), (255, 0, 0), 2)
                if barcode.data:
                    barcode_data = barcode.data.decode("utf-8")
                    barcode_type = barcode.type
                    print("Barcode Data:", barcode_data)
                    print("Barcode Type:", barcode_type)
                    move_mechArm(barcode_data)
                    # time.sleep(6)

    cv2.imshow("Barcode Detection", frame)

In [32]:
if __name__ == "__main__":
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()

        if ret:
            barcodeReader(frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

Barcode Data: 9784567524216
Barcode Type: EAN13
Moving!
Barcode Data: 9784567524216
Barcode Type: EAN13
Moving!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barcode is blank/corrupted!
Barcode Not Detected or your barc